# Пример — Линейная регрессия методом градиентного спуска

Данный notebook демонстрирует **образцовое оформление** лабораторной работы.
Используйте его как шаблон при выполнении lab1–lab4.

**Реализация модели:** [`model.py`](model.py)  
**Утилита фиксации seed:** [`../utils/seed.py`](../utils/seed.py)

---

### 1. Импорт зависимостей и фиксация случайности

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Добавляем корневую директорию labs/ в путь поиска модулей,
# чтобы импортировать общие утилиты (utils/seed.py)
sys.path.insert(0, str(Path("..").resolve()))

from utils.seed import set_global_seed
from model import LinearRegressionGD, mse

# Фиксация всех источников случайности
SEED = 42
set_global_seed(SEED)

# Директория для сохранения графиков
FIGURES_DIR = Path("results/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Seed зафиксирован: {SEED}")
print(f"Графики будут сохранены в: {FIGURES_DIR.resolve()}")

### 2. Генерация синтетических данных

Создаём данные с известной линейной зависимостью:

> y = 3·x + 7 + ε, где ε ~ N(0, σ²)

Это позволяет проверить, насколько точно модель восстанавливает
истинные параметры.

In [ ]:
# Параметры генеративной модели
TRUE_WEIGHT = 3.0
TRUE_BIAS = 7.0
NOISE_STD = 2.0
N_SAMPLES = 100

# Генерация данных
X = np.random.uniform(low=0.0, high=10.0, size=N_SAMPLES)
noise = np.random.normal(loc=0.0, scale=NOISE_STD, size=N_SAMPLES)
y = TRUE_WEIGHT * X + TRUE_BIAS + noise

# Преобразование X в матрицу (n, 1) для модели
X_matrix = X.reshape(-1, 1)

print(f"Размер X: {X_matrix.shape}")
print(f"Размер y: {y.shape}")
print(f"Первые 5 значений X: {X[:5].round(2)}")
print(f"Первые 5 значений y: {y[:5].round(2)}")

### 3. Обучение модели

Создаём экземпляр `LinearRegressionGD` и обучаем на сгенерированных данных.

In [ ]:
# Гиперпараметры
LEARNING_RATE = 0.01
N_ITERATIONS = 1000

# Создание и обучение модели
model = LinearRegressionGD(
    learning_rate=LEARNING_RATE,
    n_iterations=N_ITERATIONS,
)
model.fit(X_matrix, y)

# Найденные параметры
print("=" * 50)
print("Результаты обучения")
print("=" * 50)
print(f"Найденный вес:      {model.weights[0]:.4f}  (истинный: {TRUE_WEIGHT})")
print(f"Найденное смещение: {model.bias:.4f}  (истинное: {TRUE_BIAS})")
print(f"Финальный MSE:      {model.loss_history[-1]:.4f}")

### 4. Оценка качества модели

In [ ]:
# Предсказания модели
y_pred = model.predict(X_matrix)

# Вычисление MSE
train_mse = mse(y, y_pred)
print(f"MSE на обучающей выборке: {train_mse:.4f}")
print(f"Теоретический минимум MSE (дисперсия шума): {NOISE_STD**2:.4f}")

### 5. Визуализация: аппроксимирующая прямая

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Исходные данные
ax.scatter(X, y, alpha=0.6, color="steelblue", label="Данные", edgecolors="white")

# Аппроксимирующая прямая
X_line = np.linspace(X.min(), X.max(), 200)
y_line = model.weights[0] * X_line + model.bias
ax.plot(X_line, y_line, color="crimson", linewidth=2, label="Модель")

# Истинная зависимость
y_true_line = TRUE_WEIGHT * X_line + TRUE_BIAS
ax.plot(
    X_line, y_true_line,
    color="green", linewidth=2, linestyle="--",
    label="Истинная зависимость",
)

ax.set_xlabel("Признак x", fontsize=12)
ax.set_ylabel("Целевая переменная y", fontsize=12)
ax.set_title("Линейная регрессия: аппроксимация данных", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "regression_fit.png", dpi=150, bbox_inches="tight")
print(f"График сохранён: {FIGURES_DIR / 'regression_fit.png'}")
plt.show()

### 6. Визуализация: кривая обучения (динамика MSE)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

epochs = range(1, len(model.loss_history) + 1)
ax.plot(epochs, model.loss_history, color="darkorange", linewidth=1.5)

# Горизонтальная линия — теоретический минимум
ax.axhline(
    y=NOISE_STD**2, color="green", linestyle="--", linewidth=1,
    label=f"Дисперсия шума (σ² = {NOISE_STD**2})",
)

ax.set_xlabel("Эпоха", fontsize=12)
ax.set_ylabel("MSE", fontsize=12)
ax.set_title("Кривая обучения: MSE по эпохам", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "loss_curve.png", dpi=150, bbox_inches="tight")
print(f"График сохранён: {FIGURES_DIR / 'loss_curve.png'}")
plt.show()

### 7. Итоги

| Параметр | Истинное значение | Найденное значение |
|----------|:-----------------:|:------------------:|
| Вес (w)  | 3.0               | ≈ 3.0              |
| Смещение (b) | 7.0           | ≈ 7.0              |

Модель успешно восстановила параметры генеративной модели.
Кривая обучения демонстрирует монотонное убывание MSE,
что подтверждает корректность реализации градиентного спуска.

---

**Подробный отчёт:** [`report.md`](report.md)